In [ ]:
# ============================================
# CELL 1: Check GPU and System Info
# ============================================
!nvidia-smi
print("\n" + "="*50)
print("GPU is available!" if __import__('torch').cuda.is_available() else "⚠️ WARNING: No GPU detected!")
print("="*50)

In [ ]:
# ============================================
# CELL 2: Install System Dependencies
# ============================================
print("📦 Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq redis-server ffmpeg
print("✅ System dependencies installed!")

In [ ]:
# ============================================
# CELL 3: Clone VIDRA Project from GitHub
# ============================================
import os

print("📁 Setting up VIDRA project...\n")

# Check if already cloned
if os.path.exists('/content/Vidra-Your-AI-Video-Director'):
    print("✅ VIDRA project already exists!")
    %cd /content/Vidra-Your-AI-Video-Director
    !git pull  # Update to latest version
else:
    # Clone from GitHub
    print("📥 Cloning VIDRA from GitHub...\n")
    %cd /content
    !git clone https://github.com/Safwan2003/Vidra-Your-AI-Video-Director.git
    
    # Check if clone was successful
    if os.path.exists('/content/Vidra-Your-AI-Video-Director'):
        %cd /content/Vidra-Your-AI-Video-Director
        print("\n✅ VIDRA project cloned successfully!")
        print(f"📂 Current directory: {os.getcwd()}")
        !ls -la
    else:
        print("❌ Error: Failed to clone repository")
        print("Please check your internet connection and try again")

In [ ]:
# ============================================
# CELL 4: Install Python Dependencies
# ============================================
print("📦 Installing Python packages (this may take 5-10 minutes)...\n")

# Install backend dependencies
print("Installing backend packages...")
!pip install -q fastapi pydantic uvicorn redis
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate diffusers safetensors
!pip install -q opencv-python-headless imageio imageio-ffmpeg av
!pip install -q Pillow scipy numpy

# Install Celery explicitly
print("\nInstalling Celery...")
!pip install -q celery

# TTS: Skip Coqui TTS (not compatible with Python 3.12)
# We'll use gTTS (Google Text-to-Speech) as alternative
print("\nInstalling TTS (Text-to-Speech) - using gTTS...")
!pip install -q gTTS==2.5.1

# Install MusicGen
print("\nInstalling AudioCraft (MusicGen)...")
!pip install -q audiocraft==1.3.0

# Install LLM client
print("\nInstalling OpenAI client...")
!pip install -q openai

# Install frontend dependencies
print("\nInstalling frontend packages...")
!pip install -q streamlit requests

# Install tunneling tools
print("\nInstalling Cloudflared...")
!pip install -q cloudflared

# Install dotenv
!pip install -q python-dotenv

print("\n✅ All dependencies installed!")
print("\n⚠️ NOTE: Using gTTS instead of XTTS (Coqui TTS not compatible with Python 3.12)")
print("   Voice quality will be simpler but functional.")

In [ ]:
# ============================================
# CELL 5: Configure Environment Variables
# ============================================
import os

print("⚙️ Configuring environment...\n")

# Set your API key here (REQUIRED for storyboard generation)
# Get FREE key from: https://console.groq.com
API_KEY = ""  # ⬅️ PASTE YOUR GROQ API KEY HERE

if not API_KEY:
    print("⚠️ WARNING: No API key set!")
    print("Get a FREE key from: https://console.groq.com")
    print("Then paste it in the API_KEY variable above and re-run this cell.")
    print("\nContinuing with mock storyboard generation...\n")
else:
    os.environ['GROQ_API_KEY'] = API_KEY
    print("✅ API key configured!\n")

# Set other environment variables
os.environ['REDIS_URL'] = 'redis://localhost:6379/0'
os.environ['OUTPUT_DIR'] = './outputs'
os.environ['TORCH_DTYPE'] = 'float16'

# Create output directories
!mkdir -p outputs/images outputs/videos outputs/audio outputs/final

print("✅ Environment configured!")

In [ ]:
# ============================================
# CELL 6: Start Redis Server
# ============================================
import subprocess
import time

print("🚀 Starting Redis server...")
!redis-server --daemonize yes
time.sleep(2)

# Test Redis connection
import redis
try:
    r = redis.Redis(host='localhost', port=6379, db=0)
    r.ping()
    print("✅ Redis is running!")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")

In [ ]:
# ============================================
# CELL 7: Start Celery Worker (Background)
# ============================================
import os
import time

print("🚀 Starting Celery worker...")

# Use shell command to start Celery in background - set PYTHONPATH in the command
!cd /content/Vidra-Your-AI-Video-Director && PYTHONPATH=/content/Vidra-Your-AI-Video-Director nohup celery -A backend.main.celery_app worker --loglevel=info --pool=solo > /tmp/celery.log 2>&1 &

time.sleep(5)

print("✅ Celery worker started!")
print("   Working directory: /content/Vidra-Your-AI-Video-Director")
print("   (Processing video generation tasks in background)")
print("\n💡 Check logs with: !tail -100 /tmp/celery.log")

In [ ]:
# ============================================
# CELL 8: Start FastAPI Backend (Background)
# ============================================
import time
import os

print("🚀 Starting FastAPI backend...")

# Start FastAPI with nohup (background) - set PYTHONPATH in the command
!cd /content/Vidra-Your-AI-Video-Director && PYTHONPATH=/content/Vidra-Your-AI-Video-Director nohup uvicorn backend.main:app --host 0.0.0.0 --port 8000 > /tmp/fastapi.log 2>&1 &

# Wait for startup
print("   Waiting for FastAPI to initialize...")
time.sleep(10)

# Test backend
print("\n🔍 Testing backend connection...")
import requests
max_retries = 5
for i in range(max_retries):
    try:
        response = requests.get('http://localhost:8000/', timeout=2)
        print(f"✅ FastAPI backend running on http://localhost:8000")
        print(f"   Response: {response.json()}")
        break
    except Exception as e:
        if i < max_retries - 1:
            print(f"   ⏳ Attempt {i+1}/{max_retries} failed, retrying...")
            time.sleep(2)
        else:
            print(f"\n❌ Backend not responding after {max_retries} attempts")
            print(f"   Error: {e}")
            print("\n📋 Backend logs:")
            !tail -50 /tmp/fastapi.log

In [ ]:
# ============================================
# CELL 9: Start Streamlit with Public URL
# ============================================
import subprocess
import time
import re
import os

print("🚀 Starting Streamlit UI...\n")

# Change to project directory
os.chdir('/content/Vidra-Your-AI-Video-Director')

# Start Streamlit in background
!nohup streamlit run frontend/streamlit_app.py --server.port 8501 --server.headless true > /tmp/streamlit.log 2>&1 &

# Give Streamlit time to boot
time.sleep(8)

# Create public URL with Cloudflared
print("🌐 Creating public URL...\n")

# Start cloudflared tunnel
cf_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Parse URL from cloudflared output
public_url = None
start = time.time()
while time.time() - start < 30:
    line = cf_process.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    # Look for the public URL (*.trycloudflare.com)
    m = re.search(r'https?://[\w\.-]+\.trycloudflare\.com\S*', line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("="*60)
    print("🎉 VIDRA IS READY!")
    print("="*60)
    print(f"\n🔗 Access VIDRA at: {public_url}\n")
    print("📝 Click the link above to open the UI in a new tab")
    print("\n💡 Tips:")
    print("   - First video will take 5-10 minutes (models need to load)")
    print("   - Subsequent videos are faster")
    print("   - Keep this notebook running while generating videos")
    print("   - GPU session lasts 12 hours (free tier)")
    print("\n⚡ Quick test prompt:")
    print("   'A peaceful sunrise over mountains with birds flying'")
    print("="*60)
else:
    print("❌ Failed to create public URL")
    print("   Check logs: !tail -50 /tmp/streamlit.log")

In [ ]:
# ============================================
# CELL 10: Check Service Logs (Optional)
# ============================================
# Run this cell if you encounter errors

print("📊 Service Logs:\n")
print("=" * 60)
print("CELERY WORKER LOGS:")
print("=" * 60)
!tail -50 /tmp/celery.log

print("\n" + "=" * 60)
print("FASTAPI BACKEND LOGS:")
print("=" * 60)
!tail -50 /tmp/fastapi.log

print("\n" + "=" * 60)
print("STREAMLIT UI LOGS:")
print("=" * 60)
!tail -50 /tmp/streamlit.log

In [ ]:
# ============================================
# CELL 11: View Generated Videos (Optional)
# ============================================
# Run this to preview generated videos in the notebook

import os
from IPython.display import Video, display

print("📁 Generated Videos:\n")

final_dir = 'outputs/final'
if os.path.exists(final_dir):
    videos = [f for f in os.listdir(final_dir) if f.endswith('.mp4')]
    if videos:
        print(f"✅ {len(videos)} video(s) generated:\n")
        # Display latest video
        latest = os.path.join(final_dir, sorted(videos)[-1])
        print(f"🎬 Latest video: {os.path.basename(latest)}\n")
        display(Video(latest, width=800))
    else:
        print("   No videos generated yet.")
else:
    print("   No output directory found.")

In [ ]:
# ============================================
# CELL 12: Download Video (Optional)
# ============================================
# Run this to download the latest video to your computer

import os
from google.colab import files

final_dir = 'outputs/final'
if os.path.exists(final_dir):
    videos = [f for f in os.listdir(final_dir) if f.endswith('.mp4')]
    if videos:
        latest = os.path.join(final_dir, sorted(videos)[-1])
        print(f"📥 Downloading: {os.path.basename(latest)}")
        files.download(latest)
        print("✅ Download complete!")
    else:
        print("❌ No videos to download. Generate a video first!")
else:
    print("❌ Output directory not found.")

In [ ]:
# ============================================
# CELL 13: Stop All Services (Cleanup)
# ============================================
# Run this when you're done to free up resources

print("🛑 Stopping all services...\n")

# Kill processes
!pkill -f streamlit
!pkill -f uvicorn
!pkill -f celery
!pkill -f cloudflared
!pkill -f redis-server

print("✅ All services stopped!")
print("\n💡 To restart, run cells 6-9 again.")

---

## 🎯 Troubleshooting

### Problem: "No GPU detected"
**Solution**: Go to Runtime → Change runtime type → Select T4 GPU → Save

### Problem: "Backend not responding"
**Solution**: 
- Run Cell 10 to check logs
- Make sure Cell 6 (Redis) completed successfully
- Re-run Cells 7-8 in sequence

### Problem: "Out of memory"
**Solution**: 
- Use shorter videos (30s instead of 60s)
- Restart runtime (Runtime → Restart runtime)
- Upgrade to Colab Pro for more RAM

### Problem: "Video generation fails"
**Solution**: 
- Check Cell 10 for error logs
- Make sure API key is set in Cell 5
- Ensure GPU is enabled

---

## 📊 Performance Notes

**Expected Generation Times** (on free T4 GPU):
- Storyboard: 5-10 seconds
- Images (4 scenes): 2-3 minutes
- Animation (4 scenes): 5-8 minutes
- Audio: 30 seconds
- Final edit: 20 seconds

**Total: 8-12 minutes for a 30-second video**

---

**Made with ❤️ by the VIDRA team**